# Layer 1 — Descriptive Pass

Automated overview of all indicators in the project's processed data.

**What this notebook produces:**
- Univariate profiles (distribution, min/max/median, missing %, outliers)
- Time series trends (direction, rate of change, trend lines)
- Changepoint detection (sharp shifts, year and magnitude)
- Pairwise correlations (Pearson and Spearman, flagged above threshold)
- Geographic variation (if geo-level data exists)

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pipeline.config import load_config, get_data_processed_dir

# --- Set your project name here ---
PROJECT_NAME = "qol-immigration"  # UPDATE per project

config = load_config(PROJECT_NAME)
processed_dir = get_data_processed_dir(config)
print(f"Project: {config.get('title', PROJECT_NAME)}")
print(f"Processed data: {processed_dir}")

In [ ]:
# Load all processed Parquet files
parquet_files = sorted(processed_dir.glob("*.parquet"))
datasets = {}
for pf in parquet_files:
    datasets[pf.stem] = pd.read_parquet(pf)
    print(f"{pf.stem}: {datasets[pf.stem].shape}")

if not datasets:
    print("No processed data found. Run 'python run.py <project> --stage data' first.")

## Univariate Profiles

In [ ]:
from pipeline.analyze import univariate_profiles

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    profiles = univariate_profiles(df)
    display(profiles)

## Time Series Trends

In [ ]:
from pipeline.analyze import time_series_trends

# Specify the time column name used in your data
TIME_COL = "year"  # UPDATE if your data uses a different column

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        print(f"{name}: no '{TIME_COL}' column, skipping trends.")
        continue
    print(f"\n--- {name} ---")
    trends = time_series_trends(df, time_col=TIME_COL)
    display(trends["summary"])
    for fig in trends["figures"]:
        fig.show()

## Changepoint Detection

In [ ]:
from pipeline.analyze import detect_changepoints

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    cp_results = detect_changepoints(df, time_col=TIME_COL)
    display(cp_results["summary"])
    for fig in cp_results["figures"]:
        fig.show()

## Pairwise Correlations

In [ ]:
from pipeline.analyze import pairwise_correlations

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    corr_results = pairwise_correlations(df)
    display(corr_results["summary"])
    corr_results["heatmap"].show()

## Geographic Variation

In [ ]:
from pipeline.analyze import geographic_variation

# Specify the geographic column name used in your data
GEO_COL = "state"  # UPDATE if your data uses county, district, etc.

for name, df in datasets.items():
    if GEO_COL not in df.columns:
        print(f"{name}: no '{GEO_COL}' column, skipping geo variation.")
        continue
    print(f"\n--- {name} ---")
    geo_results = geographic_variation(df, geo_col=GEO_COL)
    display(geo_results["summary"])
    for fig in geo_results["figures"]:
        fig.show()